In [1]:
%load_ext autoreload
%autoreload 2
import torch
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from dataclasses import dataclass, field

torch.set_grad_enabled(False)  # avoid blowing up mem
device = "cuda"

In [ ]:
model_id = "google/paligemma2-3b-pt-224"
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map=device
).eval()
processor = PaliGemmaProcessor.from_pretrained(model_id)

In [ ]:
image_url = "https://github.com/zazamrykh/PicFinder/blob/main/images/doge.jpg?raw=true"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content))
plt.axis("off")
_ = plt.imshow(image)

In [4]:
@dataclass
class State:
    outputs: list[torch.Tensor] = field(default_factory=list)
    attn_weights: list[torch.Tensor] = field(default_factory=list)

    def hook(self, module, input, output):
        output_tensor, attn_weights, cache = output
        self.outputs.append(output_tensor.clone())
        self.attn_weights.append(attn_weights.clone())

In [ ]:
text = "<image>Answer en what color is the dog?"
inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=100)
response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
tokens = [processor.decode(id) for id in inputs.input_ids[0]]
print(response)

In [ ]:
layer_idx = 0
attn_layer = model.language_model.model.layers[layer_idx].self_attn
state = State()
print("state is empty:", state.outputs == [])
hook_handle = attn_layer.register_forward_hook(state.hook)
outputs = model(**inputs, output_attentions=True)
n_tokens = outputs[0].shape[1]
print("state: is empty:", state.outputs == [])
hook_handle.remove()

In [ ]:
state.outputs[0].shape, state.attn_weights[0].shape, inputs.input_ids.shape

In [ ]:
from transformers.models.gemma2.modeling_gemma2 import Gemma2Attention

attn_layer

In [ ]:
batch_size_, n_img_tokens, hidden_dim_ = model.get_image_features(
    inputs.pixel_values
).shape
grid_side_len = n_img_tokens**0.5
print(grid_side_len)
assert grid_side_len.is_integer()
grid_side_len = int(grid_side_len)


# Img Tokens Are Attending To Text Tokens

In [ ]:
# Select heads by their img attention
# hypothesis: The attentions sum up to 1 for every destintation token. Correct :)
# higher entropy means more spread out attentions. This is more interesting than dirac-like attention.
import matplotlib.pyplot as plt
from scipy.stats import entropy

# entropy([0.1, 0.1, 0.8, 0.0]), entropy([0.3, 0.3, 0.3, 0.1])
# I'll split attn into 4 quadrants: img-self attns, img-cross attns, text-self attns, text-cross attns
# the img-self attns should not be causal, and we apply the entropy measure to them
# the img-cross attns should be all zero, because they cant attend to the text
# the text-self attns should be causal
# the text-cross attentions should be dense, because they attend to the img
attn = state.attn_weights[0][0]
img_self_attn = attn[:, :n_img_tokens, :n_img_tokens]
img_cross_attn = attn[:, :n_img_tokens, n_img_tokens:]
text_self_attn = attn[:, n_img_tokens:, n_img_tokens:]
text_cross_attn = attn[:, n_img_tokens:, :n_img_tokens]

attn[0][:10, :n_img_tokens].sum(dim=1)
# # img-self attn
# entropy(img_self_attn.flatten(end_dim=1), axis=0)

# # text-self attn
# entropy(text_self_attn.flatten(end_dim=1), axis=0)

In [ ]:
import numpy as np

y = x = np.arange(n_tokens)
X, Y = np.meshgrid(x, y)

import plotly.express as px
import plotly.graph_objects as go

attn_data = attn[3].float().cpu().numpy()
fig = px.imshow(attn_data, labels=dict(x="Source Token", y="Destination Token"))
fig.update_layout(
    coloraxis_colorbar=dict(title="Attention"),
    xaxis=dict(tickmode='array', tickvals=list(range(len(tokens))), ticktext=tokens),
    yaxis=dict(tickmode='array', tickvals=list(range(len(tokens))), ticktext=tokens)
)
fig.show()

# Create Multimodal Attention Visualization

In [12]:
n_heads = 6
viz_data = dict(
    attention=state.attn_weights[0][0].round(decimals=6)[:n_heads].cpu().tolist(),
    tokens=tokens,
    image=image_url,
    image_grid_dims=[grid_side_len, grid_side_len],
    image_tokens_start=0,
    max_value=state.attn_weights[0].max().item() / 100,
    min_value=state.attn_weights[0].min().item(),
)

In [ ]:
from circuitsvis.attention import attention_heads

html = attention_heads(**viz_data)
html_str = str(html)
print(type(html_str), len(html_str))
with open("attention_heads.html", "w") as f:
    f.write(html_str)